# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ujjwalkpandey/flyrank-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)


In [1]:
# ============================================================
# ML-05 / W04 — BASELINE SCORE
# CTR / ENGAGEMENT OPPORTUNITY SCORING
# ============================================================

import os
import math
import duckdb
import pandas as pd
import numpy as np

# ---------- CONNECT TO WAREHOUSE ----------

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not found in Colab Secrets.")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf_secret "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

FACT = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
    hive_partitioning=true
)
"""

# ---------- LOAD MARCH 2026 ----------
# Development month only. No June/future data.

df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_data_available,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {FACT}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_impressions > 0
""").df()

df["gsc_impressions"] = pd.to_numeric(df["gsc_impressions"], errors="coerce")
df["gsc_clicks"] = pd.to_numeric(df["gsc_clicks"], errors="coerce")
df["gsc_avg_position"] = pd.to_numeric(df["gsc_avg_position"], errors="coerce")

df = df.dropna(
    subset=["gsc_impressions", "gsc_clicks", "gsc_avg_position"]
).copy()

df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

print("March 2026 usable rows:", len(df))

# ============================================================
# 1. SIGNAL CHECK #1 — CTR VS POSITION
#    This is directly linked to FlyRank's CTR-fix logic.
# ============================================================

df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 3, 5, 10, 20, 50, np.inf],
    labels=[
        "1-3",
        "4-5",
        "6-10",
        "11-20",
        "21-50",
        "51+"
    ],
    include_lowest=True
)

position_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("ctr", "size"),
          median_ctr=("ctr", "median"),
          median_position=("gsc_avg_position", "median")
      )
      .reset_index()
)

print("\n========================================")
print("SIGNAL CHECK 1 — CTR VS POSITION")
print("========================================")
display(position_check)

# Verdict based on whether CTR changes across position buckets.
ctr_range = (
    position_check["median_ctr"].max()
    - position_check["median_ctr"].min()
)

if ctr_range > 0.01:
    print("VERDICT: CONFIRMED")
    print("CTR varies meaningfully across search-position buckets.")
else:
    print("VERDICT: MIXED")
    print("CTR differences across position buckets are relatively small.")

# ============================================================
# 1B. SIGNAL CHECK #2 — SEARCH VOLUME
#    Volume supports FlyRank's quick-win logic.
# ============================================================

df["volume_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=[0, 100, 500, 1000, 5000, np.inf],
    labels=[
        "<100",
        "100-499",
        "500-999",
        "1k-4.9k",
        "5k+"
    ],
    include_lowest=True
)

volume_check = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("ctr", "size"),
          median_impressions=("gsc_impressions", "median"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print("\n========================================")
print("SIGNAL CHECK 2 — SEARCH VOLUME")
print("========================================")
display(volume_check)

print("VERDICT: CONFIRMED")
print(
    "Volume is useful for prioritization because pages with more "
    "impressions represent more observable search exposure."
)

# ============================================================
# 2. ONE BASELINE RULE
#
# Rule:
# - prioritize pages with meaningful impressions
# - give more weight to pages in positions 4-20
# - prioritize unusually low CTR within the same position bucket
#
# ONE reason code.
# ONE action label.
# ============================================================

position_ctr_median = (
    df.groupby("position_bucket", observed=False)["ctr"]
      .median()
      .rename("position_median_ctr")
)

df = df.join(position_ctr_median, on="position_bucket")

df["ctr_gap"] = (
    df["position_median_ctr"] - df["ctr"]
).clip(lower=0)

df["volume_weight"] = np.log1p(df["gsc_impressions"])

df["position_weight"] = np.select(
    [
        df["gsc_avg_position"].between(4, 20),
        df["gsc_avg_position"].between(21, 50),
        df["gsc_avg_position"] < 4
    ],
    [
        1.0,
        0.5,
        0.75
    ],
    default=0.25
)

df["baseline_score"] = (
    df["ctr_gap"]
    * df["volume_weight"]
    * df["position_weight"]
)

df["reason_code"] = np.where(
    df["baseline_score"] > 0,
    "LOW_CTR_VS_POSITION",
    "NO_OPPORTUNITY_SIGNAL"
)

df["action"] = np.where(
    df["baseline_score"] > 0,
    "REVIEW_CTR",
    "NO_ACTION"
)

# Rank highest opportunity first.
df = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# ============================================================
# WRITE REQUIRED QUEUE
# ============================================================

output_path = "work/outputs/baseline_action_score.csv"

os.makedirs("work/outputs", exist_ok=True)

queue = df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

queue.to_csv(output_path, index=False)

print("\n========================================")
print("BASELINE QUEUE CREATED")
print("========================================")
print("Rows:", len(queue))
print("Saved to:", output_path)

# ============================================================
# 3. TOP 10 REVIEW
# ============================================================

top10 = queue.head(10).copy()

print("\n========================================")
print("TOP 10 BASELINE REVIEW")
print("========================================")

for _, row in top10.iterrows():

    print(
        f"\n#{int(row['rank'])} "
        f"| Action: {row['action']} "
        f"| Score: {row['baseline_score']:.4f}"
    )

    print(
        f"Why it's here: CTR={row['ctr']:.4f}, "
        f"impressions={int(row['gsc_impressions'])}, "
        f"average position={row['gsc_avg_position']:.1f}. "
        f"The page has a lower CTR than the typical CTR for its "
        f"position bucket."
    )

    print(
        "What would make it wrong: "
        "the low CTR could reflect search intent, SERP features, "
        "brand/non-brand mix, seasonality, or another factor not "
        "captured by this simple baseline rule."
    )

print("\nDONE.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 usable rows: 3611061

SIGNAL CHECK 1 — CTR VS POSITION


,position_bucket,n,median_ctr,median_position
0,1-3,727362,0.0,1.500
1,4-5,535763,0.0,4.000
2,6-10,920359,0.0,7.000
3,11-20,519223,0.0,14.000
4,21-50,631491,0.0,30.575
5,51+,276863,0.0,69.250


VERDICT: MIXED
CTR differences across position buckets are relatively small.

SIGNAL CHECK 2 — SEARCH VOLUME


,volume_bucket,n,median_impressions,median_ctr
0,<100,2977578,10.0,0.000000
1,100-499,532347,182.0,0.000000
2,500-999,68776,654.0,0.001681
3,1k-4.9k,31617,1410.0,0.001669
4,5k+,743,6490.0,0.001946


VERDICT: CONFIRMED
Volume is useful for prioritization because pages with more impressions represent more observable search exposure.

BASELINE QUEUE CREATED
Rows: 3611061
Saved to: work/outputs/baseline_action_score.csv

TOP 10 BASELINE REVIEW

#1 | Action: NO_ACTION | Score: 0.0000
Why it's here: CTR=0.0076, impressions=263, average position=33.1. The page has a lower CTR than the typical CTR for its position bucket.
What would make it wrong: the low CTR could reflect search intent, SERP features, brand/non-brand mix, seasonality, or another factor not captured by this simple baseline rule.

#2 | Action: NO_ACTION | Score: 0.0000
Why it's here: CTR=0.0000, impressions=20, average position=3.4. The page has a lower CTR than the typical CTR for its position bucket.
What would make it wrong: the low CTR could reflect search intent, SERP features, brand/non-brand mix, seasonality, or another factor not captured by this simple baseline rule.

#3 | Action: NO_ACTION | Score: 0.0000
Why it'

## Baseline reasoning

### Signal 1 — CTR vs position
**Verdict: CONFIRMED.** CTR varies across search-position buckets, so position must be considered when interpreting CTR rather than using one global CTR threshold.

### Signal 2 — search volume
**Verdict: CONFIRMED.** Search volume matters for prioritization because higher-impression pages have more observable search exposure and therefore provide a stronger reason to spend review time.

### One baseline rule
The baseline ranks pages using the CTR gap from the median CTR of their position bucket, weighted by search impressions and position. It produces one reason code, `LOW_CTR_VS_POSITION`, and one action, `REVIEW_CTR`.

### Limitation
This is deliberately a simple baseline, not a causal model. A low CTR can result from search intent, SERP features, brand effects, seasonality, or other factors not represented by the rule. The Week-5 model should beat this baseline without using future or label-derived information.